# DataHub Error Case 04 — Kafka Consumer Error

**Monitor:** DataHub Kafka Consumer Error  
**Component:** datahub-gms (embedded MAE/MCE consumers)  
**Scenario:** prerequisites-kafka-controller pod shutdown → GMS loses Kafka connection

## Flow
```
prerequisites-kafka-controller scaled to 0
  → GMS embedded consumers lose Kafka connection
  → ConsumerCoordinator WARN: offset commit cannot be completed
  → KafkaHealthChecker ERROR: Kafka Health Check Failed
  → Monitor 3 fires when count > 5
```

## ⚠️ WARNING
```
Kafka down → GMS metadata processing stops (~2-3 min)
All Kafka messages preserved — no data loss
Consumers auto-rejoin after Kafka restarts
```

In [1]:
import sys
sys.path.insert(0, '.')
import importlib, datahub_test_utils
importlib.reload(datahub_test_utils)
from datahub_test_utils import *

# Verify GMS + Kafka running
kubectl('get pods -n datahub-hynix | grep -E "gms|kafka"')
r = health_check()
print(f'GMS health: {r.status_code}')
session = get_session()
print('Logged in')

✅ DataHub test utils loaded
   GMS: http://192.168.139.2:8080
   OpenSearch: https://192.168.194.1:9200
✅ DataHub test utils loaded
   GMS: http://192.168.139.2:8080
   OpenSearch: https://192.168.194.1:9200
datahub-datahub-gms-5fb84555df-xzcqf                          1/1     Running     0               24m
prerequisites-kafka-controller-0                              1/1     Running     0               13m

GMS health: 200
Logged in


In [2]:
# Cell 2: Verify Kafka consumers active before shutdown
client = os_client()
result = client.search(
    index=LOG_INDEX,
    body={
        'query': {
            'bool': {
                'must': [
                    {'range': {'@timestamp': {'gte': 'now-5m'}}},
                    {'term': {'kubernetes.container_name.keyword': 'datahub-gms'}},
                ],
                'should': [
                    {'match': {'message': 'MCL medianLag'}},
                    {'match': {'message': 'partitions assigned'}},
                    {'match': {'message': 'KafkaThrottleSensor'}},
                ],
                'minimum_should_match': 1,
            }
        },
        'sort': [{'@timestamp': {'order': 'desc'}}],
        'size': 3,
    }
)
print(f'Kafka consumers active: {result["hits"]["total"]["value"]} logs')
print_logs(result['hits']['hits'])

Kafka consumers active: 5 logs
✅ Found 3 log entries:

  [2026-06-11T01:11:03] INFO  | datahub-gms | c.d.m.d.throttle.KafkaThrottleSensor
         MCL medianLag: {MCL_VERSIONED_LAG=0, MCL_TIMESERIES_LAG=0}

  [2026-06-11T01:10:03] INFO  | datahub-gms | c.d.m.d.throttle.KafkaThrottleSensor
         MCL medianLag: {MCL_VERSIONED_LAG=0, MCL_TIMESERIES_LAG=0}

  [2026-06-11T01:09:03] INFO  | datahub-gms | c.d.m.d.throttle.KafkaThrottleSensor
         MCL medianLag: {MCL_VERSIONED_LAG=0, MCL_TIMESERIES_LAG=0}



In [3]:
# Cell 3: Shutdown Kafka
print('Scaling prerequisites-kafka-controller to 0...')
kubectl('scale statefulset prerequisites-kafka-controller --replicas=0 -n datahub-hynix')
print('Waiting 15s for Kafka to go down...')
import time; time.sleep(15)

# Verify Kafka pod gone
kubectl('get pods -n datahub-hynix | grep kafka')

Scaling prerequisites-kafka-controller to 0...
statefulset.apps/prerequisites-kafka-controller scaled

Waiting 15s for Kafka to go down...

stderr: 


CompletedProcess(args='kubectl get pods -n datahub-hynix | grep kafka', returncode=1, stdout='', stderr='')

In [4]:
# Cell 4: Find Kafka error logs — confirmed logger: KafkaThrottleSensor
wait_for_logs(5)
print('Searching for Kafka errors (last 5 min)...')
client = os_client()
result = client.search(
    index=LOG_INDEX,
    body={
        'query': {
            'bool': {
                'must': [
                    {'range': {'@timestamp': {'gte': 'now-5m'}}},
                    {'term': {'kubernetes.container_name.keyword': 'datahub-gms'}},
                    {'term': {'level.keyword': 'ERROR'}},
                ],
                'should': [
                    {'match': {'message': 'Error fetching consumer group offsets'}},
                    {'match': {'logger': 'KafkaThrottleSensor'}},
                    {'match': {'message': 'Kafka Health Check Failed'}},
                ],
                'minimum_should_match': 1,
            }
        },
        'sort': [{'@timestamp': {'order': 'desc'}}],
        'size': 10,
    }
)
print_logs(result['hits']['hits'])

⏳ Waiting 5s for logs to reach OpenSearch...
Searching for Kafka errors (last 5 min)...
✅ Found 2 log entries:

  [2026-06-11T01:15:03] ERROR | datahub-gms | c.d.m.d.throttle.KafkaThrottleSensor
         Error fetching consumer group offsets.

  [2026-06-11T01:14:03] ERROR | datahub-gms | c.d.m.d.throttle.KafkaThrottleSensor
         Error fetching consumer group offsets.



In [5]:
# Cell 5: Restore Kafka
print('Restoring prerequisites-kafka-controller...')
kubectl('scale statefulset prerequisites-kafka-controller --replicas=1 -n datahub-hynix')
print('Waiting for Kafka to start (~30s)...')
import time; time.sleep(30)

kubectl('get pods -n datahub-hynix | grep kafka')

Restoring prerequisites-kafka-controller...
statefulset.apps/prerequisites-kafka-controller scaled

Waiting for Kafka to start (~30s)...
prerequisites-kafka-controller-0                              1/1     Running     0               29s



CompletedProcess(args='kubectl get pods -n datahub-hynix | grep kafka', returncode=0, stdout='prerequisites-kafka-controller-0                              1/1     Running     0               29s\n', stderr='')

In [6]:
# Cell 6: Verify GMS consumers rejoined
print('Waiting for consumers to rejoin...')
import time; time.sleep(20)

client = os_client()
result = client.search(
    index=LOG_INDEX,
    body={
        'query': {
            'bool': {
                'must': [
                    {'range': {'@timestamp': {'gte': 'now-3m'}}},
                    {'term': {'kubernetes.container_name.keyword': 'datahub-gms'}},
                ],
                'should': [
                    {'match': {'message': 'partitions assigned'}},
                    {'match': {'message': 'Setting offset for partition'}},
                    {'match': {'message': 'Throttle exponential backoff reset'}},
                ],
                'minimum_should_match': 1,
            }
        },
        'sort': [{'@timestamp': {'order': 'desc'}}],
        'size': 3,
    }
)
print(f'Consumer recovery logs: {result["hits"]["total"]["value"]}')
print_logs(result['hits']['hits'])

r = health_check()
print(f'GMS health: {r.status_code}')

Waiting for consumers to rejoin...
Consumer recovery logs: 6
✅ Found 3 log entries:

  [2026-06-11T01:20:03] INFO  | datahub-gms | c.d.m.d.throttle.KafkaThrottleSensor
         Throttle exponential backoff reset.

  [2026-06-11T01:20:03] INFO  | datahub-gms | c.d.m.d.throttle.KafkaThrottleSensor
         Throttle exponential backoff reset.

  [2026-06-11T01:19:03] INFO  | datahub-gms | c.d.m.d.throttle.KafkaThrottleSensor
         Throttle exponential backoff reset.

GMS health: 200


In [8]:
print('\n=== Summary ===')
print('Error:   Kafka consumer group offset fetch failed')
print('Cause:   prerequisites-kafka-controller pod down')
print('Monitor: DataHub Kafka Consumer Error → fires when count > 5 in 5 min')
print('Logger:  c.d.m.d.throttle.KafkaThrottleSensor')
print('Message: Error fetching consumer group offsets')
print('Fix:     kubectl scale statefulset prerequisites-kafka-controller --replicas=1')
print('Note:    Consumers auto-rejoin after Kafka restarts')
print('         KafkaThrottleSensor fires every ~60s while Kafka is down')


=== Summary ===
Error:   Kafka consumer group offset fetch failed
Cause:   prerequisites-kafka-controller pod down
Monitor: DataHub Kafka Consumer Error → fires when count > 5 in 5 min
Logger:  c.d.m.d.throttle.KafkaThrottleSensor
Message: Error fetching consumer group offsets
Fix:     kubectl scale statefulset prerequisites-kafka-controller --replicas=1
Note:    Consumers auto-rejoin after Kafka restarts
         KafkaThrottleSensor fires every ~60s while Kafka is down
